# Coursework 1

This notebook is intended to be used as a starting point for your experiments. The instructions can be found in the MLP2025_26_CW1_Spec.pdf (see Learn,  Assignment Submission, Coursework 1). The methods provided here are just helper functions. If you want more complex graphs such as side by side comparisons of different experiments you should learn more about matplotlib and implement them. Before each experiment remember to re-initialize neural network weights and reset the data providers so you get a properly initialized experiment. For each experiment try to keep most hyperparameters the same except the one under investigation so you can understand what the effects of each are.

## Training Boilerplate

Use the below code as a boilerplate to start your experiments. You can add more cells or change the code as you see fit.

In [ ]:
from typing import Tuple

from matplotlib.figure import Figure
from pandas import DataFrame
from torch.distributions.constraints import dependent

from mlp.penalties import L1Penalty, L2Penalty, L1L2MixPenalty

ROOT = "/Users/matthewgiles/PycharmProjects/mlpractical"
FIGURE_PATH = f"{ROOT}/report/figures"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging
import sys
sys.path.append(ROOT)

from mlp.data_providers import MNISTDataProvider, EMNISTDataProvider
from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, CustomActivationLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser

In [ ]:
import os

# Set all the env variables
os.environ["MLP_DATA_DIR"] = os.path.join(ROOT, "data")

In [ ]:


from mlp.layers import DropoutLayer


class CustomModel(MultipleLayerModel):
    
    def __init__(self, 
                input_dim=784, 
                output_dim=47,
                width=128,
                depth=2,
                reg=None, 
                coefficient=0.5, 
                coefficient2=0.5,
                rng=None,
                activation=None,
             ):
        
        
        # DEPTH INCLUDES OUTPUT LAYER
        
        # Set the model parameters
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.width = width
        self.depth = depth
        self.reg = reg
        self.coefficient = coefficient
        self.coefficient2 = coefficient2
        self.rng = rng
        
        # Weight initializers    
        self.weights_init = GlorotUniformInit(rng=self.rng)
        self.biases_init = ConstantInit(0.)
        
        self.penalty = None
        self.activation = activation
        
        if self.activation is None:
            self.activation = ReluLayer()
        
        # Ensure reg is lowercase for comparison
        if self.reg is not None:
            self.reg = self.reg.lower()
                
        if reg == "l1":
            self.penalty = L1Penalty(coefficient=self.coefficient)
        elif reg == "l2":
            self.penalty = L2Penalty(coefficient=self.coefficient)
        elif reg == "l1l2":
            self.penalty = L1L2MixPenalty(self.coefficient, self.coefficient2)
        
        # Build the model
        self.build()
        self.compile()
    
    def build(self):
        
        # Init the layers
        # Hidden layer 1
        layers = []
        
        for i in range(self.depth):
            
            is_input = (i == 0)
            is_output = (i == self.depth - 1)
            
            # Get the input width
            input_width = self.input_dim if is_input else self.width
            output_width = self.output_dim if is_output else self.width
            
            print("Affine({}, in={}, out={})".format(i+1, input_width, output_width))
            
            layers.append(AffineLayer(
                input_width, output_width,
                self.weights_init, self.biases_init, 
                weights_penalty=self.penalty, biases_penalty=None,
            ))
            layers.append(self.activation)
            
            if self.reg == "dropout" and not is_output:
                layers.append(DropoutLayer(incl_prob=self.coefficient))
        
        return super().__init__(layers)
    
    def compile(self, learning_rate=9e-4):
        self.error = CrossEntropySoftmaxError()
        self.learning_rule = AdamLearningRule(learning_rate=learning_rate)
        
    def fit(self, train_data, valid_data, num_epochs=100, stats_interval=1, notebook=True, gradients=False):

        # Return a pandas dataframe with all the info we are interested in
        optimiser = Optimiser(
            self, 
            self.error, 
            self.learning_rule, 
            train_data,
            valid_data,
            data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}, 
            notebook=notebook
        )
        
        # Train the model
        stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)
        
        # Get the info
        # Set the columns to the keys of info + keys
        out_df = pd.DataFrame(columns=list(self.info.keys()) + ["epoch"] + list(keys.keys()))
        
        for e in range(num_epochs):
            
            # Copy the info
            info = self.info.copy()
            info["epoch"] = e + 1
            
            # Loop through the keys and add all their info
            for key in keys:
                info[key] = stats[e, keys[key]]
                
            # Add to the dataframe
            # AttributeError: 'DataFrame' object has no attribute 'append' 
            # out_df = out_df.append(info, ignore_index=True)
            out_df = pd.concat([out_df, pd.DataFrame([info])], ignore_index=True)

        # Order by the epoch
        out_df = out_df.sort_values(by=["epoch"])
                
        # Get the gradient plot
        if gradients:
            
            # Plot the gradient flow
            grad_plot, grad_ax = optimiser.plot_grad_flow()
            
            return out_df, grad_plot
        else:
            return out_df
    
    @property
    def info(self):
        return {
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
            "width": self.width,
            "depth": self.depth,
            "reg": self.reg,
            "coefficient": self.coefficient,
            "coefficient2": self.coefficient2,
        }


In [46]:

class Plotter:

    @staticmethod
    def plot(df, name, x_axis="epoch"):
        plt.style.use("ggplot")
    
        metrics = ["acc", "error"]
        names = ["Accuracy", "Error"]
    
        for metric, metric_name in zip(metrics, names):
            fig = plt.figure(figsize=(8, 4))
            ax = fig.add_subplot(111)
    
            # Loop over depth (or whatever `name` represents)
            for i, ind_key in enumerate(df[name].unique()):
                sub_df = df[df[name] == ind_key]
    
                # dataset type → solid vs dashed
                datasets = ["train", "valid"]
                line_style = ["-", "--"]
    
                for ds, ls in zip(datasets, line_style):
                    col = f"{metric}({ds})"
                    if col not in sub_df.columns:
                        continue
    
                    ax.plot(
                        sub_df[x_axis],
                        sub_df[col],
                        linestyle=ls,
                        linewidth=1.5,
                        label=f"{name} {ind_key}({ds})"
                    )
    
            ax.set_xlabel("Epoch number")
            ax.set_ylabel(metric_name)
            ax.legend(loc="best")
            ax.grid(True)
            fig.tight_layout()
    
            out_path = os.path.join(FIGURE_PATH, f"{name}_{metric}.png")
            print("Saving:", out_path)
            fig.savefig(out_path, bbox_inches="tight")
            plt.show()
            plt.close(fig)

    @staticmethod
    def print_latex_rows(df, cols=None, suffix=""):
        
        all_cols = cols + ["acc(valid)", "error(train)", "error(valid)"]
        
        # Get the final epoch
        final_epoch = df["epoch"].max()
        final_df = df[df["epoch"] == final_epoch]
        
        # Sort by the indepdent variable
        final_df = final_df.sort_values(by=cols)
        
        
        # Output the first row which is the col names
        
        # indepdent var & Val. Acc. & Train Error & Val. Error\\
        for _, row in final_df.iterrows():
            line = suffix
            for i, col in enumerate(all_cols):
                if i == 0:
                    line += f"{row[col]}   "
                else:
                    line += f"& {row[col]:>7.3f}   "
            line += "\\\\"
            print(line)
        

In [ ]:
# The below code will set up the data providers, random number
# generator and logger objects needed for training runs. As
# loading the data from file take a little while you generally
# will probably not want to reload the data providers on
# every training run. If you wish to reset their state you
# should instead use the .reset() method of the data providers.

# Seed a random number generator
seed = 111020
rng = np.random.RandomState(seed)
batch_size = 100
N_EPOCHS = 100

# Set up a logger object to print info about the training run to stdout
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

# Create data provider objects for the MNIST data set
train_data = EMNISTDataProvider('train', batch_size=batch_size, rng=rng)
valid_data = EMNISTDataProvider('valid', batch_size=batch_size, rng=rng)

smooth_train_data = EMNISTDataProvider('train', batch_size=batch_size, rng=rng, smooth_labels=True)
smooth_valid_data = EMNISTDataProvider('valid', batch_size=batch_size, rng=rng, smooth_labels=True)

# Question 2:

In [ ]:
widths = [128, 64, 32]

# The joined df
q2_df = pd.DataFrame()

for width in widths:
    
    # Build the model with the width
    model = CustomModel(
        width=width,
        depth=2,
    )
    model.compile(learning_rate=9e-4)
    
    # Train to get the info
    df = model.fit(
        train_data,
        valid_data,
        num_epochs=N_EPOCHS
    )
    
    # Add to the df
    q2_df = pd.concat([q2_df, df], ignore_index=True)


In [ ]:
# Plot the values
Plotter.plot(q2_df, name="width")
Plotter.print_latex_rows(q2_df, cols=["width"])

# Question 3:

In [ ]:
depths = [2, 3, 4]

# The joined df
q3_df = pd.DataFrame()

for depth in depths:
    
    # Build the model with the width
    model = CustomModel(
        depth=depth,
        width=128,
    )
    model.compile(learning_rate=9e-4)
    
    # Train to get the info
    df = model.fit(
        train_data,
        valid_data,
        num_epochs=N_EPOCHS
    )
    
    # Add to the df
    q3_df = pd.concat([q3_df, df], ignore_index=True)
    

In [ ]:
# Plot the values
Plotter.plot(q3_df, name="depth")

In [ ]:
Plotter.print_latex_rows(q3_df, cols=["depth"])

# Question 4:

In [43]:
import pandas as pd

regs = {
    "baseline": [""],
    "dropout": [0.6, 0.7, 0.85, 0.97],
    "l1": [5e-4, 1e-3, 5e-3, 5e-2],
    "l2": [5e-4, 1e-3, 5e-3, 5e-2],
    "smooth": [""],
}


q4_df = pd.DataFrame()

for reg, params in regs.items():
    
    for param in params:
        
        model = CustomModel(
            reg=reg,
            coefficient=param,
            depth=4,
        )
        model.compile(learning_rate=1e-4)
        
        # Whether we are training on smooth
        smooth = (reg == "smooth")
        
        # Train on the data
        df = model.fit(
            smooth_train_data if smooth else train_data,
            smooth_valid_data if smooth else valid_data,
            num_epochs=N_EPOCHS
        )
        
        # Add to the df
        q4_df = pd.concat([q4_df, df], ignore_index=True)
        

Epoch 72: 12.9s to complete
    error(train)=1.39e+00, acc(train)=7.89e-01, error(valid)=1.51e+00, acc(valid)=7.47e-01


  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 73: 5.5s to complete
    error(train)=1.38e+00, acc(train)=7.90e-01, error(valid)=1.51e+00, acc(valid)=7.48e-01


  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 74: 5.1s to complete
    error(train)=1.38e+00, acc(train)=7.91e-01, error(valid)=1.51e+00, acc(valid)=7.48e-01


...

Epoch 98: 5.3s to complete
    error(train)=1.36e+00, acc(train)=7.98e-01, error(valid)=1.51e+00, acc(valid)=7.45e-01


  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 99: 5.3s to complete
    error(train)=1.35e+00, acc(train)=8.01e-01, error(valid)=1.51e+00, acc(valid)=7.47e-01


  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 100: 5.3s to complete
    error(train)=1.35e+00, acc(train)=8.00e-01, error(valid)=1.51e+00, acc(valid)=7.46e-01
/var/folders/p3/gzg9876x5cdf6httpsjkn3940000gn/T/ipykernel_34844/2418060859.py:122: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out_df = pd.concat([out_df, pd.DataFrame([info])], ignore_index=True)


In [ ]:

# A plot with dropout values along the x axis and their final epoch's loss



In [ ]:
# # --- L1/L2 plot ---
# l1l2_df = df[df["regularisation"].isin(["l1", "l2"])]
# 
# plt.figure(figsize=(6, 4))
# for reg in ["l1", "l2"]:
#     sub_df = l1l2_df[l1l2_df["regularisation"] == reg]
#     plt.plot(sub_df["parameter"], sub_df["valid_acc"], marker="o", label=f"{reg.upper()} Validation Accuracy")
#     plt.plot(sub_df["parameter"], sub_df["gen_gap"], marker="s", label=f"{reg.upper()} Gen Gap")
# 
# plt.xscale("log")
# plt.xlabel("Weight Penalty (log scale)")
# plt.ylabel("Metric Value")
# plt.title("Accuracy and Error by Weight Penalty")
# plt.legend()
# plt.grid(True, which="both")
# plt.tight_layout()
# plt.savefig(os.path.join(FIGURE_PATH, f"wd_plot.png"), dpi=300)
# plt.close()

In [48]:

# Loop through the different regs
for reg in regs.keys():
    
    # Get the sub df
    sub_df = q4_df[q4_df["reg"] == reg]
    
    print(f"--- LaTeX rows for {reg} ---")
    Plotter.print_latex_rows(sub_df, cols=["coefficient"], suffix="&    ")
    print("\n")


--- LaTeX rows for baseline ---
&       &   0.763   &   0.676   &   0.918   \\


--- LaTeX rows for dropout ---
&    0.6   &   0.799   &   1.472   &   1.507   \\
&    0.7   &   0.832   &   0.902   &   0.947   \\
&    0.85   &   0.851   &   0.414   &   0.497   \\
&    0.97   &   0.849   &   0.332   &   0.509   \\


--- LaTeX rows for l1 ---
&    0.0005   &   0.763   &   0.810   &   0.824   \\
&    0.001   &   0.701   &   1.064   &   1.084   \\
&    0.005   &   0.020   &   3.850   &   3.850   \\
&    0.05   &   0.020   &   3.850   &   3.850   \\


--- LaTeX rows for l2 ---
&    0.0005   &   0.748   &   0.823   &   0.949   \\
&    0.001   &   0.842   &   0.451   &   0.539   \\
&    0.005   &   0.772   &   0.833   &   0.852   \\
&    0.05   &   0.332   &   2.567   &   2.568   \\


--- LaTeX rows for smooth ---
&       &   0.747   &   1.352   &   1.508   \\




In [45]:

q4_df.to_csv(os.path.join(FIGURE_PATH, "q4_results.csv"), index=False)
# Save the 

# Question 5:

In [ ]:

l1s = [1e-11, 1e-9]
l2s = [1e-11, 1e-9]

q5_df = pd.DataFrame()

for l1 in l1s:
    for l2 in l2s:
        
        # Build the custom model
        model = CustomModel(
            reg="l1l2",
            coefficient=l1,
            coefficient2=l2,
            depth=4,
        )
        model.compile(learning_rate=1e-4)
        
        # Train on the data
        df = model.fit(
            train_data,
            valid_data,
            num_epochs=N_EPOCHS
        )    
        
        # Add to the df
        q5_df = pd.concat([q5_df, df], ignore_index=True)


In [ ]:
Plotter.print_latex_rows(q5_df, cols=["coefficient", "coefficient2"])

# Question 6:

In [ ]:
# Build the custom model
model = CustomModel(
    activation=CustomActivationLayer(),
    depth=5,
)
model.compile(learning_rate=1e-4)

# Train on the data
q6_df, grad_plot = model.fit(
    train_data,
    valid_data,
    gradients=True,
    num_epochs=N_EPOCHS
)

# Save the grad plot
Plotter.plot(q6_df, name="custom_activation")
grad_plot.savefig(os.path.join(FIGURE_PATH, f"custom_activation_grad_flow.png"), bbox_inches="tight")



In [ ]:
raise Exception("Stop here")